In [19]:
import re
from lxml import etree
import html
import os
import glob # Import glob to find files

# --- Configuration ---
# Directory containing your XML files ( ADJUST AS NEEDED )
# Use '../' to go up one level if needed, '.' for current
SOURCE_DIR = '../GRC_misc/'

# Glob pattern to find the desired XML files ( ADJUST AS NEEDED )
# Examples:
# 'aesch.*.headlam_eng2.xml' # Specific Aeschylus Headlam
# '*.some_author.*.xml'      # All works by an author
# '*.*grc*.xml'              # All Greek files
# '*.*eng*.xml'              # All English files
#FILE_PATTERN = 'aesch.*.headlam_eng2.xml' # <-- CHANGE THIS PATTERN AS NEEDED
FILE_PATTERN = 'xenophon01.watson_1854.xml'

# Base directory for all HTML outputs
BASE_OUTPUT_DIR = './html_output/'

# --- Dynamic Output Directory ---
# Creates a sub-folder based on the first part of the pattern
# e.g., 'headlam' from 'aesch.*.headlam_eng2.xml'
# You can customize this logic or set a specific name
pattern_parts = FILE_PATTERN.split('.')
output_subfolder = pattern_parts[2] if len(pattern_parts) > 3 else 'default_output'
OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, output_subfolder)

# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Find XML files using the specified pattern and source directory
XML_FILES = glob.glob(os.path.join(SOURCE_DIR, FILE_PATTERN))

# XML namespaces
ns = {'tei': 'http://www.tei-c.org/ns/1.0',
      'xml': 'http://www.w3.org/XML/1998/namespace'} # Add xml namespace

if not XML_FILES:
    print(f"Warning: No XML files found in '{SOURCE_DIR}' matching '{FILE_PATTERN}'")
else:
    print(f"Found {len(XML_FILES)} XML files in '{SOURCE_DIR}' matching '{FILE_PATTERN}':")
    # Sort for consistent order
    XML_FILES.sort()
    for f in XML_FILES:
        print(f"  - {os.path.basename(f)}")
print(f"HTML output will be saved in '{OUTPUT_DIR}'.")

Found 1 XML files in '../GRC_misc/' matching 'xenophon01.watson_1854.xml':
  - xenophon01.watson_1854.xml
HTML output will be saved in './html_output/default_output'.


In [20]:
from collections import defaultdict
import html
from lxml import etree

def first_ancestor_n(el, subtype):
    # nearest ancestor div of given subtype
    hit = el.xpath(f"ancestor::tei:div[@type='textpart' and @subtype='{subtype}'][1]", namespaces=ns)
    return hit[0].get('n') if hit else None

def preceding_section_milestone_n(p):
    # look for an immediately preceding milestone @unit='section'
    prev = p.getprevious()
    if prev is not None and isinstance(prev.tag, str):
        if etree.QName(prev.tag).localname == 'milestone' and prev.get('unit') == 'section':
            return prev.get('n')
    return None

# per (book, chapter) section counter (starts at 0, increment on first paragraph with no explicit @n)
section_counter = defaultdict(int)



def build_inner_html(parent_element):
    """Recursively serialize a TEI element’s text and children into inline HTML."""
    parts = []
    if parent_element.text:
        parts.append(html.escape(parent_element.text))
    for child in parent_element.iterchildren():
        tagname = etree.QName(child).localname
        # Simple inline serialization for text-level tags
        if tagname in {"hi", "q", "title", "term", "foreign", "ref"}:
            inner = build_inner_html(child)
            parts.append(f"<{tagname}>{inner}</{tagname}>")
        else:
            # fallback: raw HTML serialization for unhandled tags
            child_html = etree.tostring(child, encoding="unicode", method="html", with_tail=False)
            parts.append(child_html)
        if child.tail:
            parts.append(html.escape(child.tail))
    return "".join(parts).strip()

def render_div(el):
    """Renders a div, handling its header and ALL children (p, sp, castList, tables, etc.)."""
    parts = []

    # Resolve book/chapter context for this div
    book_n = first_ancestor_n(el, 'book') or el.get('n') if el.get('subtype') == 'book' else first_ancestor_n(el, 'book')
    chap_n = first_ancestor_n(el, 'chapter') or el.get('n') if el.get('subtype') == 'chapter' else first_ancestor_n(el, 'chapter')

    # --- 1. RENDER THE HEADER ---
    head = el.find('tei:head', ns)
    if head is not None:
        head_html = build_inner_html(head).strip()
        if head_html:
            if book_n and chap_n:
                chap_label = f"{book_n}.{chap_n}"
                chap_id = chap_label.replace('.', '-')
                parts.append(f'<h3><a class="loc" id="loc-{chap_id}">{chap_label}</a> {head_html}</h3>')
            else:
                # Use h2 for main titles (like intro), h3 for others
                title_tag = head.find("tei:title[@type='main']", ns)
                if title_tag is not None:
                     parts.append(f"<h2 class='index-main'>{head_html}</h2>")
                else:
                     parts.append(f"<h3>{head_html}</h3>")
    
    # --- 2. RENDER ALL CHILDREN (p, sp, castList, table, sub-divs, etc.) ---
    for element in el.iterchildren():
        if not isinstance(element.tag, str): # Skip comments
            continue 
        
        tag = etree.QName(element.tag).localname
        
        # Skip the head tag, we already processed it
        if tag == 'head':
            continue

        # --- Re-use logic from the main cell's loop ---
        if tag == 'p':
            p_html = build_inner_html(element)
            attrs = " ".join([f'{k}="{v}"' for k,v in element.attrib.items() if '{' not in k])
            
            b = first_ancestor_n(element, 'book') or book_n
            c = first_ancestor_n(element, 'chapter') or chap_n
            s = element.get('n') or preceding_section_milestone_n(element)
            if not s and (b and c):
                section_counter[(b, c)] += 1
                s = str(section_counter[(b, c)])
            
            if b and c and s:
                label = f"{b}.{c}.{s}"
                pid = label.replace('.', '-')
                parts.append(f'<p{" "+attrs if attrs else ""}><a class="loc" id="loc-{pid}">{label}</a> {p_html}</p>')
            else:
                parts.append(f"<p{' '+attrs if attrs else ''}>{p_html}</p>")
        
        elif tag == 'sp':
             sp_inner_html_parts = []
             speaker_tag = element.find('tei:speaker', ns)
             if speaker_tag is not None:
                 speaker_text = "".join(speaker_tag.itertext()).strip()
                 start_line = speaker_tag.get('start-line')
                 end_line = speaker_tag.get('end-line')
                 line_range_text = ""
                 if start_line and end_line:
                     line_range_text = f" [{start_line}–{end_line}]"
                 elif start_line:
                     line_range_text = f" [Line {start_line}]"
                 
                 if speaker_text:
                     full_speaker_display = f"{html.escape(speaker_text)}{line_range_text}" 
                     sp_inner_html_parts.append(f"<span class=\"speaker\">{full_speaker_display}</span>")
                 
                 # Temporarily remove speaker to avoid double serialization by build_inner_html
                 # Note: This modifies the element, but it's ok for this script
                 speaker_tag.getparent().remove(speaker_tag) 

             sp_inner_content = build_inner_html(element)
             sp_inner_html_parts.append(sp_inner_content)
             final_sp_html = "".join(sp_inner_html_parts).strip()
             if final_sp_html:
                 parts.append(f"<div class=\"speech\">{final_sp_html}</div>")

        elif tag == 'castList':
            cl_head = element.find('tei:head', ns)
            if cl_head is not None:
                parts.append(f"<h4>{html.escape(''.join(cl_head.itertext()).strip())}</h4>")
            parts.append("<ul class=\"cast-list\">")
            for item in element.findall('tei:castItem', ns):
                role_tag = item.find('tei:role', ns)
                role_text = ""
                desc_text = ""
                if role_tag is not None:
                    role_text = "".join(role_tag.itertext()).strip()
                    if role_tag.tail:
                        desc_text = role_tag.tail.strip().lstrip(',').strip()
                else:
                    desc_text = "".join(item.itertext()).strip()
                if role_text:
                    parts.append(f"<li><span class=\"role\">{html.escape(role_text)}</span> <span class=\"role-desc\">{html.escape(desc_text)}</span></li>")
                elif desc_text:
                    parts.append(f"<li><span class=\"role-desc\">{html.escape(desc_text)}</span></li>")
            parts.append("</ul>")

        elif tag == 'l':
             l_inner_html = build_inner_html(element)
             if l_inner_html:
                 parts.append(f'<p class="verse-line">{l_inner_html}</p>')
        
        elif tag == 'hr' or tag == 'span': # Handle pre-processed pb tags
             element_html = etree.tostring(element, encoding='unicode', method='html', with_tail=False).strip()
             parts.append(element_html)
             if element.tail:
                  parts.append(html.escape(element.tail))
        
        elif tag == 'table': # Handle tables (for index)
             table_html = build_inner_html(element)
             parts.append(f"<table>{table_html}</table>")
        
        elif tag == 'ab': # Handle <ab> tag (for index)
             ab_html = build_inner_html(element)
             parts.append(f"<p>{ab_html}</p>") # Treat as paragraph

        elif tag == 'div':
            # Recurse for nested divs
            parts.append(render_div(element))
        
        else: # Fallback
            print(f"    - Info: render_div fallback for tag: {tag}")
            element_html = etree.tostring(element, encoding='unicode', method='html', with_tail=False).strip()
            if element_html:
               parts.append(element_html)
               if element.tail:
                    parts.append(html.escape(element.tail))
    
    return "".join(parts)


def render_index_div(el):
    """Render an index div (now just calls render_div)"""
    # render_div is now powerful enough to handle indexes, tables, and all.
    return render_div(el)

In [21]:
import re
from lxml import etree
import html
import os
import glob # Import glob to find files

# XML namespaces (assuming TEI)
ns = {'tei': 'http://www.tei-c.org/ns/1.0'}

def process_play(xml_filepath, html_output_filepath):
    """
    Parses an Aeschylus Headlam TEI XML file, extracts the English text
    and footnotes, and generates a two-column HTML file.
    Handles different TEI structures and nested tags within notes.
    Implements bidirectional linking between text refs and footnotes.
    Correctly processes pb tags anywhere in the main text.
    """
    print(f"\nProcessing '{os.path.basename(xml_filepath)}'...")
    try:
        # --- NEW: Initialize sets to track keys ---
        found_note_keys = set()
        linked_ref_keys = set()
        # --- END NEW ---
        # --- 1. Parsing ---
        parser = etree.XMLParser(remove_blank_text=True, recover=True)
        root = etree.parse(xml_filepath, parser)
        print("   - XML parsed.")

        # --- 2. Data Extraction ---
        footnote_col = {} # Dictionary to hold notes, keyed by a unique identifier

        # *** PRE-PROCESSING STEP FOR NOTES ***
        all_notes = root.findall('.//tei:note', ns)
        for note in all_notes:
            # Recursively process elements within the note
            for elem in list(note.xpath('.//*')): # Use list() for safe iteration
                parent = elem.getparent()
                if parent is None: continue

                tag = etree.QName(elem.tag).localname

                if tag == 'l':
                    br = etree.Element("br")
                    br.tail = elem.tail
                    line_text = "".join(elem.itertext()).strip()
                    previous_sibling = elem.getprevious()
                    if previous_sibling is not None:
                        previous_sibling.tail = (previous_sibling.tail or '') + line_text
                    else:
                        parent.text = (parent.text or '') + line_text
                    parent.replace(elem, br)
                elif tag == 'quote':
                    bq = etree.Element("blockquote")
                    bq.text = elem.text; bq.tail = elem.tail
                    for k, v in elem.attrib.items():
                        if k == '{http://www.w3.org/XML/1998/namespace}lang': bq.set('lang', v)
                        elif '{' not in k: bq.set(k, v)
                    for child in list(elem): bq.append(child)
                    parent.replace(elem, bq)
                elif tag == 'foreign':
                    i = etree.Element("i")
                    i.text = elem.text; i.tail = elem.tail
                    for k, v in elem.attrib.items():
                        if k == '{http://www.w3.org/XML/1998/namespace}lang': i.set('lang', v)
                        elif '{' not in k: i.set(k, v)
                    for child in list(elem): i.append(child)
                    parent.replace(elem, i)
                elif tag == 'bibl':
                    strong = etree.Element("strong")
                    strong.text = elem.text; strong.tail = elem.tail
                    for k, v in elem.attrib.items():
                         if '{' not in k: strong.set(k, v)
                    for child in list(elem): strong.append(child)
                    parent.replace(elem, strong)
                elif tag == 'title':
                    i_title = etree.Element("i")
                    i_title.text = elem.text; i_title.tail = elem.tail
                    for k, v in elem.attrib.items():
                         if '{' not in k: i_title.set(k, v)
                    for child in list(elem): i_title.append(child)
                    parent.replace(elem, i_title)

        # --- ADD THIS BLOCK ---
                elif tag == 'gloss':
                    # Convert <gloss> to <span class="gloss">
                    span_gloss = etree.Element("span")
                    span_gloss.set("class", "gloss")
                    # Add title attribute for tooltip using target or rend
                    gloss_title_attr = elem.get('target', elem.get('rend', ''))
                    if gloss_title_attr:
                        span_gloss.set('title', html.escape(gloss_title_attr)) # Escape attribute value

                    # --- Critical: Transfer content ---
                    # Move the text content
                    span_gloss.text = elem.text
                    # Move all child elements
                    for child in list(elem): # Use list() for safe iteration while moving
                        span_gloss.append(child)
                    # --- End Critical ---

                    # Keep the tail (text after the gloss tag)
                    span_gloss.tail = elem.tail
                    # Replace original <gloss> with the new <span>
                    parent.replace(elem, span_gloss)
                # --- END ADDED BLOCK ---
        # ... (after the elif for 'gloss') ...

                elif tag == 'emph' or tag == 'hi': # Handle emphasis/highlighting
                    new_tag_name = 'em' if tag == 'emph' else 'i' # Use <em> for emph, <i> for hi
                    fmt_tag = etree.Element(new_tag_name)
                    fmt_tag.text = elem.text; fmt_tag.tail = elem.tail
                    # Copy attributes like rend if needed (optional)
                    # if elem.get('rend'): fmt_tag.set('class', elem.get('rend'))
                    for child in list(elem): fmt_tag.append(child)
                    parent.replace(elem, fmt_tag)

                elif tag == 'q': # Handle inline quote, could also use <blockquote> if appropriate
                    q_tag = etree.Element("q") # Using HTML <q> for inline
                    q_tag.text = elem.text; q_tag.tail = elem.tail
                    for child in list(elem): q_tag.append(child)
                    parent.replace(elem, q_tag) # Corrected: replace elem with q_tag
                elif tag == 's':  # Handle TEI <s> (sentence)
                    # In TEI, <s> marks a sentence; in HTML, <s> = strikethrough.
                    # So we must convert or unwrap it safely.
                    s_tag = etree.Element("span")  # Neutral HTML element
                    s_tag.text = elem.text
                    s_tag.tail = elem.tail
                    for child in list(elem):
                        s_tag.append(child)

                    # Optional: add a class for styling, if you want to see sentence boundaries
                    s_tag.set("class", "sentence")

                    parent.replace(elem, s_tag)  # replace the original TEI <s> with <span>
        # --- A. Extract Footnotes (Robustly) ---
        all_notes = root.findall('.//tei:note', ns)
        notes_processed_count = 0
        for note in all_notes:
            key = None
            n_attr = note.get('n')
            xml_id = note.get('{http://www.w3.org/XML/1998/namespace}id')

            # Determine the key for linking (prioritize xml_id derived key if present)
            if xml_id and xml_id.startswith('note-'):
                key = xml_id.replace('note-', '').replace('-', '.') # Format 2 (ag.xml) preferred
            elif note.get('type') == 'footnote' and n_attr and '.' in n_attr:
                key = n_attr # Format 1 (supp.xml) fallback
            elif n_attr: # Handle simple n="1" if no other key found
                 # Check if the corresponding ref n="N" exists in main text to validate
                 ref_exists = root.xpath(f'.//tei:ref[@n="{n_attr}"]', namespaces=ns)
                 if ref_exists:
                     key = n_attr
                 elif xml_id: # Fallback to raw xml_id if n doesn't seem right
                     key = xml_id
            elif xml_id: # Last resort if only xml_id exists
                 key = xml_id

            if not key:
                print(f"   - Warning: Skipping note without a usable key: {etree.tostring(note, encoding='unicode')[:50]}...")
                continue
            # --- NEW: Add key to the set ---
            found_note_keys.add(key)
            # --- END NEW ---
            # Serialize the inner HTML content of the note
            note_parts = [html.escape(note.text or '')]
            for child in note:
                note_parts.append(etree.tostring(child, encoding='unicode', method='html'))
            note_html_content = "".join(note_parts).strip()
            html_content = f'<div class="line" id="note-{key}"><a class="note-ref" data-note="{key}">{key}</a> <span class="text">{note_html_content}</span></div>'
            
            # --- NEW: Check for existing key ---
            if key in footnote_col:
                print(f"   - WARNING: Duplicate note key detected: '{key}'. Overwriting previous entry.")
                # Optional: You could store duplicates elsewhere if needed
                # duplicate_notes.setdefault(key, []).append(html_content)
            # --- END NEW ---
                
            footnote_col[key] = html_content
            notes_processed_count += 1
        # Remove the <note> tags from the main tree AFTER processing them
        all_notes_again = root.findall('.//tei:note', ns)
        for note in all_notes_again:
            parent = note.getparent()
            if parent is not None:
                if note.tail:
                    prev = note.getprevious()
                    if prev is not None:
                        prev.tail = (prev.tail or '') + note.tail
                    else:
                        parent.text = (parent.text or '') + note.tail
                parent.remove(note)

        # Sort footnotes for output
        def sort_key(n_str):
            parts = re.split(r'[.-]', n_str)
            return [p.zfill(5) if p.isdigit() else p.lower() for p in parts]
        sorted_note_keys = sorted(footnote_col.keys(), key=sort_key)
        footnote_html = "\n".join([footnote_col.get(key, '') for key in sorted_note_keys]) # Use .get for safety

        
        # --- B. Transform English Text XML for HTML ---
        #main_text_div = root.find('.//tei:div[@type="translation"]', ns)
        main_text_div = root.find('.//tei:body', ns)

        
        if main_text_div is None:
            

            main_text_div = root.find('.//tei:div[@type="textpart"]', ns)
            if not  main_text_div is None:
                print(f"   - Debug: Found main_text_div: {main_text_div.tag}") # Add this line

        if main_text_div is None:
            main_text_div = root.find('.//tei:div[@type="textpart"]', ns)
            print(f"   - Error: Could not find <div type=\"textpart\"> or <div type=\"translation\">. Skipping file.")
            return
        else:
            # If we landed on a textpart but it has no paragraph/line/speech content, try translation again
            has_content = main_text_div.xpath('.//tei:p | .//tei:l | .//tei:sp', namespaces=ns)
            if not has_content:
                alt = root.find('.//tei:div[@type="translation"]', ns)
                if alt is not None:
                    main_text_div = alt
            # --- ADD MORE DEBUGGING HERE ---
            print(f"   - Debug: main_text_div confirmed for ref search: {{http://www.tei-c.org/ns/1.0}}{main_text_div.get('type') or main_text_div.get('subtype')} n='{main_text_div.get('n')}'")
            # Try searching from root as a test
            all_refs_in_doc = root.findall('.//tei:ref', ns)
            print(f"   - Debug: Found {len(all_refs_in_doc)} <ref> tags in the *entire document* using root.findall.")

            # --- END DEBUGGING ---
        if main_text_div is None:
            print('   - Error: Could not find main text div.')
            english_html = "<p>Error: Main text content not found.</p>"
        else:

            
    
            
            # --- Pre-process main text elements ---
    # Search for refs starting from the root, as we know this finds them all
            all_refs_in_doc = root.findall('.//tei:ref', ns)
            print(f"   - Debug: Found {len(all_refs_in_doc)} refs using root.findall. Processing them...")
    
            # 1. Replace <ref n="..."> tags with <a> links
            refs_found = 0 # Will count processed refs
            refs_linked = 0
            # Iterate over the list found from the root
            for ref in list(all_refs_in_doc): # Use list for safe iteration if replacing
                # We assume any ref found via root.findall is potentially valid if it has 'n'
    
                refs_found += 1 # Tentatively count it
                n_attr = ref.get('n')
                if not n_attr:
                    refs_found -= 1 # Don't count refs without 'n'
                    continue # Skip refs without n attribute
    
                # Determine the key this ref should link to...
                actual_key_found = None
                potential_key1 = n_attr
                potential_key2 = n_attr.replace('-', '.') if '-' in n_attr else None
    
                if potential_key1 in footnote_col:
                    actual_key_found = potential_key1
                elif potential_key2 in footnote_col:
                    actual_key_found = potential_key2
    
                if actual_key_found:
                    refs_linked += 1
                    linked_ref_keys.add(actual_key_found)
    
                    a_tag = etree.Element("a")
                    a_tag.set("href", f"#note-{actual_key_found}")
                    a_tag.set("class", "fn-link")
                    a_tag.set("data-note", actual_key_found)
                    a_tag.text = actual_key_found # Use key as link text
                    a_tag.tail = ref.tail # Preserve text after the ref
    
                    parent = ref.getparent()
                    if parent is not None:
                        # Find index and replace
                        try:
                            index = parent.index(ref)
                            parent.remove(ref)
                            parent.insert(index, a_tag)
                        except ValueError:
                             print(f"   - Warning: Could not find index for ref '{n_attr}' to replace it.")
                             # As a fallback, try appending if index fails (might mess order)
                             # parent.append(a_tag)
                    else:
                        print(f"   - Warning: Ref '{n_attr}' has no parent, cannot replace.")
                else:
                    print(f"   - Warning: Ref '{n_attr}' found but no matching note key ({potential_key1} or {potential_key2}) in footnote_col.")
                    refs_found -= 1 # Decrement count if ref is found but cannot be linked
            # End of loop
    
            # 2. Replace <lb n="..."> tags with <br> tags
            for lb in list(main_text_div.findall('.//tei:lb', ns)):
                br_tag = etree.Element("br")
                br_tag.tail = lb.tail
                parent = lb.getparent()
                if parent is not None:
                    parent.replace(lb, br_tag)
    
    # *** THIS IS THE FIX ***
            # 3. Replace <pb n="..."> tags with actual etree Elements BEFORE serialization
            for pb in list(main_text_div.findall('.//tei:pb', ns)): # Use list() for safe iteration
                parent = pb.getparent()
                if parent is None: continue
    
                pb_n = pb.get('n', '')
    
                # Create the <hr> element
                hr_tag = etree.Element('hr')
                hr_tag.set('class', 'pb-hr')
                # The <hr> tag's tail will initially be empty or carry text immediately after it
    
                # Create the <span> element
                span_tag = etree.Element('span')
                span_tag.set('class', 'pb')
                span_tag.text = f"[Page {pb_n}]" if pb_n else "[Page Break]"
                # CRITICAL: The span tag carries the tail text that originally followed the <pb> tag
                span_tag.tail = pb.tail
    
                # Find insertion index
                pb_index = parent.index(pb)
    
                # Insert the new elements in order
                parent.insert(pb_index, hr_tag)      # Insert <hr> first at pb's original position
                parent.insert(pb_index + 1, span_tag) # Insert <span> immediately after <hr>
    
                # Remove the original <pb> element
                parent.remove(pb)
    
    # 4. Replace ALL <stage> tags with <span class="stage">
            for stage in list(main_text_div.findall('.//tei:stage', ns)): # Use list() for safe iteration
                parent = stage.getparent()
                if parent is None: continue
    
                # Create the new <span> element
                span_tag = etree.Element("span")
                span_tag.set("class", "stage")
                
                # Copy inner content (text and children)
                span_tag.text = stage.text
                for child in list(stage):
                    span_tag.append(child)
                
                # Copy the tail (text immediately after the tag)
                span_tag.tail = stage.tail
                
                # Replace the old <stage> tag with the new <span> tag
                parent.replace(stage, span_tag)    
    
    # (End of Section B processing)
            print(f"   - Data extracted: {notes_processed_count} footnotes processed.")
            print(f"   - Refs found: {refs_found}, Refs linked: {refs_linked}")
    
    
    # (End of Section B processing)
            print(f"   - Data extracted: {notes_processed_count} footnotes processed.")
            print(f"   - Refs found: {refs_found}, Refs linked: {refs_linked}")
    
            print(f"   - Debug C: Starting serialization. main_text_div is {'NOT None' if main_text_div is not None else '!!! NONE !!!'}")
            # --- C. Serialize Transformed XML to HTML String (Explicit Text/Tail Handling) ---
            html_builder = []
    
            #if(1): # more than 100 lines need to be unindented
    
    
    
            # Iterate through the direct children of the main_text_div AFTER pre-processing
        for element in main_text_div.iterchildren():
                    if not isinstance(element.tag, str): # Skip comments
                        continue 
                    
                    tag = etree.QName(element.tag).localname
        
                    try:
                        if tag == 'div':
                            div_subtype = element.get('subtype')
                            if div_subtype == 'index':
                                html_builder.append(render_index_div(element)) # Uses powerful new renderer
                            else:
                                html_builder.append(render_div(element)) # Uses powerful new renderer
                        else:
                            # This will catch any tags directly under <body> that are NOT divs
                            # (like the <pb> tag at the end of Xenophon's body)
                            # We can just serialize them directly
                            if tag == 'hr' or tag == 'span': # Handle pre-processed pb tags
                                 element_html = etree.tostring(element, encoding='unicode', method='html', with_tail=False).strip()
                                 html_builder.append(element_html)
                                 if element.tail:
                                      html_builder.append(html.escape(element.tail))
                            elif tag == 'p' or tag == 'ab': # Handle loose paragraphs under body
                                 p_html = build_inner_html(element)
                                 html_builder.append(f'<p>{p_html}</p>')
                            else:
                                print(f"    - Info: Main loop fallback for tag: {tag}")
                                element_html = etree.tostring(element, encoding='unicode', method='html', with_tail=False).strip()
                                if element_html:
                                   html_builder.append(element_html)
                                   if element.tail:
                                        html_builder.append(html.escape(element.tail))
                    except Exception as e:
                        print(f"    - Warning: Could not serialize element {tag} (Main Loop): {e}")    
            
                    english_html = "\n".join(html_builder) # Join the final HTML parts
                    #else:
                    #english_html = "<p>Error: Main text content not found.</p>"
            
            
                    # --- 3. HTML Generation ---
                    # (Remains unchanged...)
                    # ...
                    # --- HTML Template (2-Column) ---
                    # (Remains unchanged...)
                    # ...
                    # --- 4. Write to file ---
                    # (Remains unchanged...)
                    # ...
                    # --- 3. HTML Generation ---
                    # (Remains unchanged...)
                    header_title_tag = root.find('.//tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title', ns)
                    #main_title = "".join(header_title_tag.itertext()).strip() if header_title_tag is not None else os.path.basename(xml_filepath)
                    #author = "Aeschylus (Headlam, trans.)"
            # --- 3. HTML Generation (Dynamic Metadata) ---
                    header = root.find('.//tei:teiHeader', ns)
                    main_title = "Untitled Document"
                    author_text = "Unknown Author"
                    translator_text = ""
            
                    if header is not None:
                        # Extract Main Title
                        title_tag = header.find('.//tei:fileDesc/tei:titleStmt/tei:title', ns)
                        if title_tag is not None:
                            main_title = "".join(title_tag.itertext()).strip()
            
                        # Extract Author
                        author_tag = header.find('.//tei:fileDesc/tei:titleStmt/tei:author', ns)
                        if author_tag is not None:
                            author_text = "".join(author_tag.itertext()).strip()
            
                        # Extract Translators
                        translators = header.findall('.//tei:fileDesc/tei:titleStmt/tei:editor[@role="translator"]', ns)
                        if translators:
                            translator_names = ["".join(t.itertext()).strip() for t in translators]
                            translator_text = f" ({', '.join(translator_names)}, trans.)" # Format as "(Name, trans.)"
            
                    # Combine author and translator for display
                    author_display = f"{author_text}{translator_text}"
            
                    # Get main lang from root text element for HTML lang attribute
                    text_lang = root.find('.//tei:text', ns).get('{http://www.w3.org/XML/1998/namespace}lang', 'en')
            
                    # --- Variables for HTML Template ---
                    # (js_scroll_options, js_log_link_click, js_log_ref_click remain the same)
                    js_scroll_options = "{ behavior: 'smooth', block: 'center' }"
                    js_log_link_click = "console.log(`Clicked text link for note: ${noteId}`); // Debug"
                    js_log_ref_click = "console.log(`Clicked footnote ref for note: ${noteId}`); // Debug"
                    
            # Create the JS object string outside the f-string to avoid nested brace confusion
                    js_scroll_options = "{ behavior: 'smooth', block: 'center' }"
                    # Define JS console log strings outside the main f-string
                    js_log_link_click = "console.log(`Clicked text link for note: ${noteId}`); // Debug"
                    js_log_ref_click = "console.log(`Clicked footnote ref for note: ${noteId}`); // Debug"
                    # --- HTML Template (2-Column) ---
                    # (Remains unchanged...)
                    html_template = f"""
            <!DOCTYPE html>
                    <html lang="{text_lang}"> 
                    <head>
                        <meta charset="UTF-8">
                        <meta name="viewport" content="width=device-width, initial-scale=1.0">
                        <title>{{main_title}}</title> 
                        <style>
                        body {{ font-family: 'Georgia', serif; display: flex; flex-direction: column; height: 100vh; margin: 0; background-color: #fdfdfd; }}
                            header {{ padding: 10px 20px; border-bottom: 2px solid #ddd; background-color: #fff; text-align: center; z-index: 10; }}
                            h1 {{ margin: 0; font-size: 1.8em; color: #333; }}
                            h2 {{ margin: 5px 0 0; font-size: 1.2em; color: #666; font-style: italic; font-weight: normal;}}
                            .container {{
                                display: grid;
                                grid-template-columns: 2fr 1fr; /* 2 columns, text on left is 2x wider */
                                gap: 15px;
                                flex-grow: 1;
                                padding: 10px;
                                overflow: hidden;
                            }}
                            .column {{
                                background-color: #ffffff;
                                border: 1px solid #e0e0e0;
                                border-radius: 4px;
                                display: flex;
                                flex-direction: column;
                                overflow: hidden;
                            }}
            /* --- Style for <add> tags --- */
                            .tei-add {{
                                color: #006400; /* Dark green text */
                                /* Optional: Subtle background */
                                /* background-color: #e9f5e9; */
                                /* Optional: Slightly smaller font size? */
                                /* font-size: 0.95em; */
                            }}
                            .tei-add::before {{
                                content: '<';
                                font-weight: bold;
                                color: #006400; /* Match text color */
                                margin-right: 0.1em; /* Small space after bracket */
                            }}
                            .tei-add::after {{
                                content: '>';
                                font-weight: bold;
                                color: #006400; /* Match text color */
                                margin-left: 0.1em; /* Small space before bracket */
                            }}
                            /* --- Style for <gloss> tags --- */
                            .gloss {{
                                border-bottom: 1px dotted #555; /* Subtle dotted underline */
                                cursor: help; /* Changes cursor to indicate potential tooltip */
                                /* Optional: Slightly lighter text */
                                /* color: #444; */ 
                            }}
                            .gloss[title] {{ /* Add specific style if there's a tooltip */
                                 /* Keep the default help cursor */
                            }}
                            .loc {{ color: #666; text-decoration: none; font-family: monospace; margin-right: 0.5em; }}
                            .column > h3 {{
                                text-align: center; margin: 0; padding: 12px;
                                border-bottom: 1px solid #e0e0e0; background-color: #f9f9ff;
                                color: #444; font-size: 1em; text-transform: uppercase; letter-spacing: 0.5px;
                                position: sticky; top: 0; z-index: 5;
                            }}
                            .content {{
                                padding: 15px;
                                overflow-y: auto;
                                flex-grow: 1;
                                line-height: 1.6;
                                /* Enable smooth scrolling */
                                scroll-behavior: smooth;
                            }}
            
                            /* English Text Column Styles */
                            #english-content h2 {{ font-size: 1.4em; color: #000; font-style: normal; text-align: center; margin-bottom: 1em;}}
                            #english-content h4 {{ font-size: 1.1em; text-align: center; margin: 1em 0 0.5em; text-transform: uppercase;}}
                            #english-content ul {{ list-style-type: none; padding-left: 0; text-align: center; margin-bottom: 1em;}}
                            #english-content li {{ margin-bottom: 0.3em; }}
            
            /* New Cast List Styles */
                            #english-content .cast-list {{
                                list-style-type: none;
                                text-align: left; /* Align text to the left */
                                margin: 1.5em 3em; /* Give it some horizontal margin */
                                padding: 1.5em; /* Add padding inside the box */
                                border: 1px solid #eee;
                                background: #fdfdfd;
                                border-radius: 4px;
                            }}
                            #english-content .cast-list li {{
                                margin-bottom: 0.6em; /* Space out the list items */
                            }}
                            #english-content .cast-list .role {{
                                font-weight: bold;
                                color: #800000; /* Match the speaker color */
                            }}
                            #english-content .cast-list .role-desc {{
                                color: #333;
                            }}
                            
                            #english-content .speech {{ margin-bottom: 1em; }}
                            #english-content .speaker {{ font-weight: bold; margin-right: 8px; color: #800000; display: block; margin-bottom: 0.2em; }}
                            #english-content .pb-hr {{ border: none; border-top: 1px dashed #ccc; margin: 1em 0; }} /* Style hr for pb */
                            #english-content .pb {{ display: block; text-align: center; color: #888; font-size: 0.9em; margin-bottom: 1em; }}
                            
                            #english-content .stage {{
                                display: inline-block; /* Allows it to flow inline but accept padding/margins */
                                font-style: italic; 
                                color: #555;
                                background-color: #fafafa; /* Light gray background */
                                padding: 5px 8px;      /* Give it breathing room */
                                margin: 5px;           /* Separate it from surrounding text */
                                border: 1px solid #eee;  /* Subtle border */
                                border-radius: 3px;
                                line-height: 1.4;      /* Ensure text height is normal */
                            }}   
            
                            
                            #english-content .fn-link {{
                                vertical-align: super; font-size: 0.8em; text-decoration: none;
                                color: #007bff; font-weight: bold; padding: 0 2px;
                                line-height: 1;
                                border-radius: 2px;
                            }}
                            #english-content .fn-link:hover, #english-content .fn-link.highlight {{
                                 text-decoration: underline; background-color: #fff9c4;
                                 /* Add outline for better visibility */
                                 outline: 1px solid #fdd835;
                                 outline-offset: 1px;
                            }}
                            #english-content p {{ margin: 0.5em 0; }}
                            #english-content .verse-line {{ margin: 0.1em 0 0.1em 2em; text-indent: -1em; }}
            
            
                            /* Footnote Column Styles */
                            .line {{ display: flex; align-items: baseline; padding: 5px; border-radius: 3px; border-bottom: 1px solid #f0f0f0; }}
                            .line:last-child {{ border-bottom: none; }}
                            .line.highlight {{
                                background-color: #fff9c4;
                                /* Add outline for better visibility */
                                outline: 1px solid #fdd835;
                                outline-offset: -1px; /* Inset outline slightly */
                            }}
                            .note-ref {{
                                flex-shrink: 0; width: 60px; font-size: 0.8em; color: #555;
                                cursor: pointer; text-align: right; margin-right: 10px; font-family: monospace;
                                padding-top: 0.1em;
                            }}
                            .note-ref:hover {{ color: #007bff; text-decoration: underline; }}
                            .text {{ line-height: 1.5; font-size: 0.9em; flex-grow: 1; }}
            
                            /* Style the converted TEI tags within notes */
                            .text blockquote {{
                                font-style: italic; color: #333;
                                border-left: 3px solid #ccc;
                                padding-left: 10px; margin: 0.5em 0 0.5em 5px;
                                display: block;
                            }}
                             .text blockquote br {{
                                display: block; content: ""; margin-top: 0.2em;
                             }}
                            .text i {{ font-style: italic; }}
                            .text strong {{ font-weight: bold; }}
                            .text br {{
                                 display: block; content: ""; margin-top: 0.2em;
                            }}
            
                        </style>
            </head>
                    <body>
                        <header>
                            <h1>{main_title}</h1> 
                            <h2>{author_display}</h2> 
                        </header>
                        <div class="container">
                            <div class="column">
                                <h3>English Translation</h3>
                                <div class="content" id="english-content">{english_html}</div>
                            </div>
                            <div class="column">
                                <h3>Footnotes</h3>
                                <div class="content" id="footnote-content">{footnote_html}</div>
                            </div>
                        </div>
                        {{/* *** SCRIPT FOR BIDIRECTIONAL SCROLLING/HIGHLIGHTING *** */}}
            <script>
                                     // All curly brackets must be doubled to {{ and }} for the Python f-string
                                    document.addEventListener('DOMContentLoaded', function() {{
                                        const container = document.querySelector('.container');
                                        const englishContent = document.getElementById('english-content');
                                        const footnoteContent = document.getElementById('footnote-content');
                                        let lastHighlightedNoteElement = null;
                                        let lastHighlightedLinkElements = []; // Can be multiple links
            
                                        // Function to remove all highlights
                                        function removeHighlights() {{
                                            if (lastHighlightedNoteElement) {{
                                                lastHighlightedNoteElement.classList.remove('highlight');
                                            }}
                                            lastHighlightedLinkElements.forEach(link => {{
                                                link.classList.remove('highlight');
                                            }});
                                            lastHighlightedNoteElement = null;
                                            lastHighlightedLinkElements = [];
                                        }}
            
                                        // Function to find, scroll, and highlight elements
                                        function findScrollHighlight(targetNoteId, scrollTargetColumn) {{ // scrollTargetColumn is 'note' or 'link'
                                            removeHighlights(); // Clear previous state
            
                                            // These JS variables were fixed in a previous step and are correct
                                            const noteElement = document.getElementById(`note-${{targetNoteId}}`);
                                            const linkElements = englishContent.querySelectorAll(`.fn-link[data-note="${{targetNoteId}}"]`);
            
                                            // Highlight Note
                                            if (noteElement) {{
                                                noteElement.classList.add('highlight');
                                                lastHighlightedNoteElement = noteElement; // Store for removal
                                                // Scroll the note column if the link was clicked
                                                if (scrollTargetColumn === 'note') {{
                                                    noteElement.scrollIntoView({js_scroll_options}); // This variable is correct
                                                }}
                                            }}
            
                                            // Highlight ALL corresponding links and scroll the FIRST one
                                            if (linkElements.length > 0) {{
                                                linkElements.forEach(link => {{
                                                    link.classList.add('highlight');
                                                }});
                                                lastHighlightedLinkElements = Array.from(linkElements); // Store all for removal
                                                // Scroll the first link if the note was clicked
                                                if (scrollTargetColumn === 'link') {{
                                                    linkElements[0].scrollIntoView({js_scroll_options}); // This variable is correct
                                                }}
                                            }}
                                        }}
            
                                        // Add click listener using event delegation on the container
                                        container.addEventListener('click', function(event) {{
                                            const linkTarget = event.target.closest('.fn-link');
                                            const noteRefTarget = event.target.closest('.note-ref');
            
                                            if (linkTarget && linkTarget.dataset.note) {{
                                                event.preventDefault(); // Prevent default anchor jump behavior
                                                const noteId = linkTarget.dataset.note;
            
                                                {js_log_link_click}
                                                findScrollHighlight(noteId, 'note'); // Scroll the note column ('note' column is the target)
            
                                            }} else if (noteRefTarget && noteRefTarget.dataset.note) {{
                                                event.preventDefault(); // Prevent default anchor jump behavior
                                                const noteId = noteRefTarget.dataset.note;
                                                {js_log_ref_click}
                                                findScrollHighlight(noteId, 'link'); // Scroll the text column ('link' column is the target)
                                            }}
            
                                        }});
                                    }});
                                </script>
                    </body>
                    </html>
                    """
            
                    # --- 4. Write to file ---
                    with open(html_output_filepath, 'w', encoding='utf-8') as f:
                        f.write(html_template)
            
            
            # ... (after processing refs and notes, before final serialization or writing) ...
            
                    # --- NEW: Calculate and report unlinked notes ---
                    unlinked_note_keys = found_note_keys - linked_ref_keys
                    if unlinked_note_keys:
                        print(f"   - Warning: {len(unlinked_note_keys)} Notes found but NOT linked from text:")
                        # Sort for consistent output
                        sorted_unlinked = sorted(list(unlinked_note_keys), key=sort_key)
                        for unlinked_key in sorted_unlinked:
                            print(f"     - Note key: {unlinked_key}")
                    # --- END NEW ---
            
                    # --- C. Serialize Transformed XML to HTML String ---
                    # ... (rest of the serialization and file writing code) ...
                    print(f"   - Successfully created '{os.path.basename(html_output_filepath)}'.")

    except IOError:
        print(f"   - Error: File not found '{xml_filepath}'.")
    except etree.XMLSyntaxError as e:
        print(f"   - Error: XML syntax error in '{xml_filepath}'. {e}")
    except Exception as e:
        print(f"   - Error: An unexpected error occurred processing '{xml_filepath}'.")
        import traceback
        traceback.print_exc() # Print full error for debugging

In [22]:
# Check if the XML_FILES list (from Cell 1) exists and has files in it
if 'XML_FILES' not in locals() or not XML_FILES:
    print("Error: XML_FILES list not found or is empty.")
    print("Please make sure you have run the first cell successfully.")
else:
    print(f"\n--- Starting conversion of {len(XML_FILES)} files ---")
    
    # Loop through each file path found in Cell 1
    for xml_file in XML_FILES:
        try:
            # Create the output filename
            # e.g., 'aesch.ag.headlam_eng2.xml' -> 'aesch.ag.headlam_eng2.html'
            base_name = os.path.basename(xml_file)
            html_name = os.path.splitext(base_name)[0] + '.html'
            
            # Create the full output path using the OUTPUT_DIR from Cell 1
            html_output_file = os.path.join(OUTPUT_DIR, html_name)
            
            # Call the main function defined in Cell 2
            process_play(xml_file, html_output_file)
            
        except Exception as e:
            print(f"   - FAILED to process {xml_file}: {e}")

    print("\n--- Conversion complete ---")
    print(f"All HTML files have been saved to the '{OUTPUT_DIR}' directory.")


--- Starting conversion of 1 files ---

Processing 'xenophon01.watson_1854.xml'...
   - XML parsed.
   - Warning: Skipping note without a usable key: <note xmlns="http://www.tei-c.org/ns/1.0">The late...
   - Warning: Skipping note without a usable key: <note xmlns="http://www.tei-c.org/ns/1.0" type="fo...
   - Warning: Skipping note without a usable key: <note xmlns="http://www.tei-c.org/ns/1.0" type="fo...
   - Warning: Skipping note without a usable key: <note xmlns="http://www.tei-c.org/ns/1.0" type="fo...
   - Warning: Skipping note without a usable key: <note xmlns="http://www.tei-c.org/ns/1.0" type="fo...
   - Warning: Skipping note without a usable key: <note xmlns="http://www.tei-c.org/ns/1.0" type="fo...
   - Warning: Skipping note without a usable key: <note xmlns="http://www.tei-c.org/ns/1.0" type="fo...
   - Warning: Skipping note without a usable key: <note xmlns="http://www.tei-c.org/ns/1.0" type="fo...
   - Warning: Skipping note without a usable key: <note xmlns="http